In [2]:
"""
This performs a rate of rise cut across all available bands for the candidates in
sn_near_galaxy_candidate.csv, extragalactic_lt20mag_candidates.csv and
candidates_in_TNS.csv. It pulls the diaObjectIds from the CSV file, fetches the light
curve data from the Fink API, computes a peak-normalised pre-peak linear slope for
every band present in each object's light curve, and passes any object with at least
one band showing a nonzero pre-peak rise. This replaces the earlier g/r-only, fixed
empirical threshold version, since that was found to miss objects whose only clear
pre-peak rise was in a band other than g or r (e.g. i-band for higher redshift SNe).
"""

import pandas as pd
import numpy as np
from pathlib import Path
from light_curve_plotter import collect_light_curve_data

# Directories for the objects that passed the first filtering done through the tags, second directory is for the output of the rate of rise cut.
TEST_DIR = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data"
PLOTS_DIR = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "images" / "test_data_images"
TEST_PASSED_DIR = TEST_DIR / "test_passed.csv"
TEST_FAILED_DIR = TEST_DIR / "test_failed.csv"
# Reading the csv for the objects that passed the Fink tag filtering and creating the CSVs where data will be stored.

# Specifying how many objects go into each query to the API.
CHUNK_SIZE = 500

# Minimum number of pre-peak points needed in a band before a slope is even attempted.
# np.polyfit needs at least 2 points to fit a line; kept low deliberately while we test
# the loosened nonzero-slope criterion, per today's discussion with Anais.
MIN_PRE_PEAK_POINTS = 4

# Minimum time baseline (in days) spanned by the pre-peak points used in a fit.
# Points clustered within a very short window can produce wildly unstable slopes,
# since the denominator of the fit (variance in time) collapses toward zero even
# with many points - this was seen directly in some objects showing slopes of 10-100+
# despite having 10-20+ pre-peak detections, all crammed into a single night or two.
MIN_PRE_PEAK_TIME_SPAN_DAYS = 5

# This is so we remove spurious and noisey data that skews the results of the fit process too much
MIN_PEAK_SNR = 10

def collect_light_curve_info_with_fallback(candidates: list[str]) -> pd.DataFrame | None:
    """This function collects the lightcurve info of an object
    and factors in a recursive self repair incase the script timesout
    while running on large requests."""

    data = collect_light_curve_data(",".join(candidates))
    if data is not None:
        return data

    if len(candidates) <= 1:
        print(f"Could not fetch data for object {candidates}, skipping.")
        return None

    mid = len(candidates) // 2
    left = collect_light_curve_info_with_fallback(candidates[:mid])
    right = collect_light_curve_info_with_fallback(candidates[mid:])

    candidates_parts = [df for df in (left, right) if df is not None]
    if not candidates_parts:
        return None
    return pd.concat(candidates_parts)


def chunked_light_curve_data(candidates: list, chunk_size: int = 500) -> pd.DataFrame:
    """
    This function scrapes the Fink API for the light curve info necessary
    to make cuts based on object qualities and produce plots of the light curves.
    """
    candidate_results = []

    for diaObjectId in range(0, len(candidates), chunk_size):
        chunked_ids = candidates[diaObjectId:diaObjectId + chunk_size]
        candidates_light_curve_data = collect_light_curve_info_with_fallback(chunked_ids)
        if candidates_light_curve_data is not None:
            candidate_results.append(candidates_light_curve_data)

    if not candidate_results:
        raise RuntimeError(f"No light curve data returned for candidate batch of size {len(candidates)}")

    return pd.concat(candidate_results, ignore_index=True)


def slope_calculation(group_df: pd.DataFrame, band: str, min_points: int = MIN_PRE_PEAK_POINTS, min_time_span: int = MIN_PRE_PEAK_TIME_SPAN_DAYS, min_peak_snr: int = MIN_PEAK_SNR) -> tuple[float, int] | None:
    """
    This function finds the slope of the band of the lightcurve prepeak by fitting a
    polynomial of degree 1 to the points of the light curve before the brightest point
    of the light curve. Returns None if there isn't enough pre-peak data in this band
    to attempt a fit.

    This is done to remove the flat objects whose brightness doesn't change that much
    over time. We make it a function of bands so that we can easily use different
    bandpasses when necessary for higher redshift SNe.
    """
    band_df = group_df[(group_df["r:band"] == band) & (group_df["r:psfFlux"] > 0)]

    if band_df.empty:
        return None

    significant_df = band_df[band_df["r:psfFlux"] / band_df["r:psfFluxErr"] > MIN_PEAK_SNR]
    if significant_df.empty:
        return None

    peak_flux = significant_df["r:psfFlux"].max()
    peak_mjd = significant_df.loc[significant_df["r:psfFlux"].idxmax(), "r:midpointMjdTai"]
    pre_peak = significant_df[significant_df["r:midpointMjdTai"] < peak_mjd]

    if len(pre_peak) < min_points:
        return None

    time_span = pre_peak["r:midpointMjdTai"].max() - pre_peak["r:midpointMjdTai"].min()
    if time_span < min_time_span:
        return None

    normalised_flux = pre_peak["r:psfFlux"] / peak_flux
    normalised_err = pre_peak["r:psfFluxErr"] / peak_flux

    # Guard against zero/negative errors, which would otherwise produce an
    # infinite or undefined weight.
    normalised_err = normalised_err.clip(lower=1e-6)
    weights = 1 / normalised_err

    slope, _ = np.polyfit(pre_peak["r:midpointMjdTai"], normalised_flux, 1, w=weights)


    return slope, len(pre_peak)


def compute_all_band_slopes(group_df: pd.DataFrame, min_points: int = MIN_PRE_PEAK_POINTS) -> dict[str, tuple[float, int]]:
    """
    Computes the pre-peak slope for every band present in this object's light curve,
    not just a fixed pair. Returns a dict of band -> (slope, n_points) for every band
    where a slope could be calculated; bands with insufficient pre-peak data are
    omitted rather than included as None, so downstream code doesn't need to filter.
    """
    bands_present = group_df["r:band"].unique()
    slopes = {}

    for band in bands_present:
        result = slope_calculation(group_df, band, min_points)
        if result is not None:
            slopes[band] = result

    return slopes


def check_rate_of_rise(group_df: pd.DataFrame, min_points: int = MIN_PRE_PEAK_POINTS) -> bool:
    """
    Computes the pre-peak slope across every available band for this object, and
    passes the object if at least one band shows a nonzero (positive) pre-peak rise.

    This is deliberately loose for now: rather than requiring specific bands (as the
    old g/r-only version did) or a calibrated threshold, this just checks whether the
    object is doing *anything* pre-peak in at least one band. Tighter criteria
    (uncertainty weighting, max time-gap between points, a proper threshold) are
    intended to be layered on top of this once it's validated on a small batch.
    """
    slopes = compute_all_band_slopes(group_df, min_points)

    if not slopes:
        return False

    return any(slope > 0.001 for slope, _ in slopes.values())


def applying_the_rate_of_rise(objects: list[str], chunk_size: int, output_path_passed: Path =0, output_path_failed: Path =0) -> tuple[list[str], list[str]]:
    """
    This function pulls the light curve data for the specified list of object ids and
    the chunk size can be customised if you have a large list of objects or small. The
    function then checks whether the object shows a nonzero pre-peak rise in at least
    one band and directs the objects to their respective lists. Finally this function
    produces two CSV files of the objects that passed and ones that failed.
    """
    passed, failed = [], []

    light_curve_info = chunked_light_curve_data(objects, chunk_size)

    for object_id, group_df in light_curve_info.groupby("r:diaObjectId"):
        if check_rate_of_rise(group_df):
            passed.append(object_id)
        else:
            failed.append(object_id)

    pd.DataFrame(passed, columns=["r:diaObjectId"]).to_csv(output_path_passed, index=False)
    pd.DataFrame(failed, columns=["r:diaObjectId"]).to_csv(output_path_failed, index=False)

    return passed, failed


# if __name__ == "__main__":
#     applying_the_rate_of_rise(SN_NEAR_GALAXY_CANDIDATES, CHUNK_SIZE, SN_NEAR_GALAXY_CANDIDATES_OUTPUT_PASSED, SN_NEAR_GALAXY_CANDIDATES_OUTPUT_FAILED)
#     applying_the_rate_of_rise(EXTRAGALACTIC_CANDIDATES, CHUNK_SIZE, EXTRAGALACTIC_CANDIDATES_OUTPUT_PASSED, EXTRAGALACTIC_CANDIDATES_OUTPUT_FAILED)
#     applying_the_rate_of_rise(IN_TNS_CANDIDATES, CHUNK_SIZE, IN_TNS_CANDIDATES_OUTPUT_PASSED, IN_TNS_CANDIDATES_OUTPUT_FAILED)

test_candidates = pd.read_csv("/home/riley/Documents/University/Honours/Honours_Project/data/extragalactic_object_ids/extragalactic_lt20mag_candidate.csv", dtype={"r:diaObjectId":str})["r:diaObjectId"].to_list()

applying_the_rate_of_rise(test_candidates, 500, TEST_PASSED_DIR, TEST_FAILED_DIR)

([170028485716738057,
  170028485847285898,
  170028485999329316,
  170028489473785922,
  170028500588167170,
  170028510922408006,
  170028512375734309,
  170028513073561804,
  170028526579744777,
  170028529201184852,
  170235219334922248,
  170261592629837872,
  170411112000913487,
  170587109388189815,
  170587115712675908,
  170587117705494629,
  170591539818922118,
  170591542622814412,
  313629129709715483,
  313637935287631915,
  313637935723315254,
  313637935762112523,
  313673107053215773,
  313690716557017106,
  313690716728459265,
  313690716884172831,
  313699514821640259,
  313761042391892105,
  313761043604045880,
  313791827794198596,
  313831458447818885,
  313831458523316231,
  313853496686280764,
  313853533153132568,
  313862200773574662,
  313862204981510146,
  313871015583154191,
  313871018649190439,
  313879819574051272,
  313888627074007083,
  313936986529333309,
  313945770552721409,
  313972182535897094,
  313972183280910340,
  313985345023639575,
  31398534

In [42]:
"""
This performs a rate of rise cut across all available bands for the candidates in
sn_near_galaxy_candidate.csv, extragalactic_lt20mag_candidates.csv and
candidates_in_TNS.csv. It pulls the diaObjectIds from the CSV file, fetches the light
curve data from the Fink API, computes a peak-normalised pre-peak linear slope for
every band present in each object's light curve, and passes any object with at least
one band showing a nonzero pre-peak rise.
"""

from pathlib import Path
import numpy as np
import polars as pl
from light_curve_plotter import collect_light_curve_data

# Directories
TEST_DIR = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data"
PLOTS_DIR = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "images" / "test_data_images"
TEST_PASSED_DIR = TEST_DIR / "test_passed.csv"
TEST_FAILED_DIR = TEST_DIR / "test_failed.csv"

# Configuration
CHUNK_SIZE = 500
MIN_PRE_PEAK_POINTS = 4
MIN_PRE_PEAK_TIME_SPAN_DAYS = 5
MIN_PEAK_SNR = 10


def collect_light_curve_info_with_fallback(candidates: list[str]) -> pl.DataFrame | None:
    """Collects lightcurve info and recursively splits requests if time-out occurs."""
    data = collect_light_curve_data(",".join(candidates))
    if data is not None:
        # Convert Pandas response from API wrapper to Polars DataFrame if needed
        return pl.from_pandas(data) if not isinstance(data, pl.DataFrame) else data

    if len(candidates) <= 1:
        print(f"Could not fetch data for object {candidates}, skipping.")
        return None

    mid = len(candidates) // 2
    left = collect_light_curve_info_with_fallback(candidates[:mid])
    right = collect_light_curve_info_with_fallback(candidates[mid:])

    candidates_parts = [df for df in (left, right) if df is not None]
    if not candidates_parts:
        return None
    
    # Diagonal concat handles slight schema mismatches if present across batches
    return pl.concat(candidates_parts, how="diagonal")


def chunked_light_curve_data(candidates: list[str], chunk_size: int = 500) -> pl.DataFrame:
    """Scrapes the Fink API in chunked batches."""
    candidate_results = []

    for i in range(0, len(candidates), chunk_size):
        chunked_ids = candidates[i : i + chunk_size]
        candidates_light_curve_data = collect_light_curve_info_with_fallback(chunked_ids)
        if candidates_light_curve_data is not None:
            candidate_results.append(candidates_light_curve_data)

    if not candidate_results:
        raise RuntimeError(f"No light curve data returned for candidate batch of size {len(candidates)}")

    return pl.concat(candidate_results, how="diagonal")


def slope_calculation(
    band_df: pl.DataFrame,
    min_points: int = MIN_PRE_PEAK_POINTS,
    min_time_span: int = MIN_PRE_PEAK_TIME_SPAN_DAYS,
    min_peak_snr: int = MIN_PEAK_SNR,
) -> tuple[float, int] | None:
    """Fits degree 1 polynomial to pre-peak points of a single band in a light curve."""
    # Filter for significant points (SNR > MIN_PEAK_SNR)
    significant_df = band_df.filter((pl.col("r:psfFlux") / pl.col("r:psfFluxErr")) > min_peak_snr)
    if significant_df.is_empty():
        return None

    # Find peak flux and its corresponding MJD
    max_idx = significant_df["r:psfFlux"].arg_max()
    peak_flux = significant_df["r:psfFlux"][max_idx]
    peak_mjd = significant_df["r:midpointMjdTai"][max_idx]

    # Pre-peak data points
    pre_peak = significant_df.filter(pl.col("r:midpointMjdTai") < peak_mjd)

    if len(pre_peak) < min_points:
        return None

    mjd_vals = pre_peak["r:midpointMjdTai"].to_numpy()
    time_span = mjd_vals.max() - mjd_vals.min()
    if time_span < min_time_span:
        return None

    flux_vals = pre_peak["r:psfFlux"].to_numpy() / peak_flux
    err_vals = pre_peak["r:psfFluxErr"].to_numpy() / peak_flux

    # Guard against zero/negative errors
    err_vals = np.clip(err_vals, a_min=1e-6, a_max=None)
    weights = 1.0 / err_vals

    slope, _ = np.polyfit(mjd_vals, flux_vals, 1, w=weights)
    return slope, len(pre_peak)


def compute_all_band_slopes(
    group_df: pl.DataFrame, min_points: int = MIN_PRE_PEAK_POINTS
) -> dict[str, tuple[float, int]]:
    """Computes pre-peak slope for each available band."""
    bands_present = group_df["r:band"].unique().to_list()
    slopes = {}

    for band in bands_present:
        band_df = group_df.filter(pl.col("r:band") == band)
        result = slope_calculation(band_df, min_points=min_points)
        if result is not None:
            slopes[band] = result

    return slopes


def check_rate_of_rise(group_df: pl.DataFrame, min_points: int = MIN_PRE_PEAK_POINTS) -> bool:
    """Passes object if at least one band has a positive pre-peak rise."""
    slopes = compute_all_band_slopes(group_df, min_points)
    if not slopes:
        return False
    return any(slope > 0 for slope, _ in slopes.values())


def applying_the_rate_of_rise(
    objects: list[str],
    chunk_size: int,
    output_path_passed: Path,
    output_path_failed: Path,
) -> tuple[list[str], list[str]]:
    """Groups light curves by ID, tests rate of rise, and saves output CSVs."""
    passed, failed = [], []

    light_curve_info = chunked_light_curve_data(objects, chunk_size)

    # Iterating Polars group_by yields ((object_id,), sub_df)
    for (object_id,), group_df in light_curve_info.group_by("r:diaObjectId"):
        if check_rate_of_rise(group_df):
            passed.append(object_id)
        else:
            failed.append(object_id)

    # Polars CSV output (no index=False required)
    pl.DataFrame({"r:diaObjectId": passed}).write_csv(output_path_passed)
    pl.DataFrame({"r:diaObjectId": failed}).write_csv(output_path_failed)

    return passed, failed


if __name__ == "__main__":
    # Optimized CSV read with schema_overrides and column selection
    test_candidates = (
        pl.read_csv(
            TEST_DIR / "1000_random_test_objects.csv",
            schema_overrides={"r:diaObjectId": pl.String},
            columns=["r:diaObjectId"],
        )["r:diaObjectId"]
        .to_list()
    )

    applying_the_rate_of_rise(test_candidates, CHUNK_SIZE, TEST_PASSED_DIR, TEST_FAILED_DIR)

In [ ]:
"""Easy X match to check if objects that passed/failed are SN/non SN """
import polars as pl
from pathlib import Path

# The directories for the objects that passed and the objects that failed.
TEST_DATA_DIR = Path.home() / "Documents" / "University"/"Honours"/"Honours_Project"/"data"/"test_data"
PASSED_OBJECTS =  TEST_DATA_DIR / "test_passed.csv"
FAILED_OBJECTS = TEST_DATA_DIR / "test_failed.csv"
SN_LIKE_TEST_OBJECTS_DIR = TEST_DATA_DIR / "sn_like_test_objects.csv"
NON_SN_LIKE_TEST_OBJECTS_DIR = TEST_DATA_DIR / "non_sn_like_test_objects.csv"

# Reading the CSV files into dataframe for X matching.
PASSED_OBJECTS_DF = pl.read_csv(PASSED_OBJECTS)
FAILED_OBJECTS_DF = pl.read_csv(FAILED_OBJECTS)
# PASSED_OBJECTS_DF = pl.read_csv(PASSED_OBJECTS, schema_overrides={"r:diaObjectId": pl.String}, columns=["r:diaObjectId"])["r:diaObjectId"].to_list()
# FAILED_OBJECTS_DF = pl.read_csv(FAILED_OBJECTS, schema_overrides={"r:diaObjectId": pl.String}, columns=["r:diaObjectId"])["r:diaObjectId"].to_list()
print (len(PASSED_OBJECTS_DF))

# Truth dataframes
SN_LIKE_TEST_OBJECTS_DF = pl.read_csv(SN_LIKE_TEST_OBJECTS_DIR)
NON_SN_LIKE_TEST_OBJECTS_DF = pl.read_csv(NON_SN_LIKE_TEST_OBJECTS_DIR)

sn_that_passed = SN_LIKE_TEST_OBJECTS_DF.join(PASSED_OBJECTS_DF, on="r:diaObjectId", how="inner")
sn_that_did_not_pass = SN_LIKE_TEST_OBJECTS_DF.join(FAILED_OBJECTS_DF, on="r:diaObjectId", how="inner")
non_sn_that_passed = NON_SN_LIKE_TEST_OBJECTS_DF.join(FAILED_OBJECTS_DF, on="r:diaObjectId", how="inner")

print (sn_that_passed)
print (sn_that_did_not_pass)
print(non_sn_that_passed)

sn_that_did_not_pass = (sn_that_did_not_pass.with_columns(pl.col("r:diaObjectId").cast(pl.Utf8)))
sn_that_did_not_pass = sn_that_did_not_pass["r:diaObjectId"].to_list()
sn_that_passed = (sn_that_passed.with_columns(pl.col("r:diaObjectId").cast(pl.Utf8)))
sn_that_passed = sn_that_passed["r:diaObjectId"].to_list()

10
shape: (10, 1)
┌────────────────────┐
│ r:diaObjectId      │
│ ---                │
│ i64                │
╞════════════════════╡
│ 170019716323082348 │
│ 170032900274651160 │
│ 170032917163016282 │
│ 170046089928376424 │
│ 170235219334922248 │
│ 170261592629837872 │
│ 313699514821640259 │
│ 313835836426682441 │
│ 313945770546430083 │
│ 313963405286309936 │
└────────────────────┘
shape: (0, 1)
┌───────────────┐
│ r:diaObjectId │
│ ---           │
│ i64           │
╞═══════════════╡
└───────────────┘
shape: (10, 1)
┌────────────────────┐
│ r:diaObjectId      │
│ ---                │
│ i64                │
╞════════════════════╡
│ 170019717135204495 │
│ 170019696305242310 │
│ 170019696442081285 │
│ 170019716324130933 │
│ 170019716327800983 │
│ 170019716479320072 │
│ 170028485639143520 │
│ 170028485639143548 │
│ 170028485642813476 │
│ 170028485648056327 │
└────────────────────┘


In [3]:
import io
import requests as re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.time import Time

SLOPES_OUTPUT_PATH = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data" / "test_fitted_slopes.csv"

def append_fitted_slopes(rows: list[dict], output_path: Path = SLOPES_OUTPUT_PATH) -> None:
    """
    Appends per-band fitted slope rows to a CSV file, writing a header only if the
    file doesn't already exist. Each row is expected to contain diaObjectId, band,
    slope, and n_pre_peak_points.
    """
    if not rows:
        return

    new_rows_df = pd.DataFrame(rows)

    new_rows_df.to_csv(output_path, mode="a", header=True, index=False)

band_colours = {"u": "darkblue", "g":"blue", "r": "red", "i":"darkred", "z":"orange", "y":"black"}   

def collect_light_curve_data(diaObjectId):
    r = re.post(
        "https://api.lsst.fink-portal.org/api/v1/sources",
        json={
            "diaObjectId": diaObjectId,
            "columns": "r:diaObjectId, r:diaSourceId, r:midpointMjdTai, r:psfFlux, r:psfFluxErr, r:band",
            "output-format": "json",
        },
    )
    if r.status_code != 200:
        print (f"Failed to fetch the light curve info for {diaObjectId}: {r.status_code}")
        return None
    return pd.read_json(io.BytesIO(r.content))

def lightcurve_plotting(df: pd.DataFrame, diaObjectId: str, source_file: str, output_dir: str, min_pre_peak_points: int = 4, min_pre_peak_time_span: float = 5, slopes_output_path: Path = SLOPES_OUTPUT_PATH, min_peak_SNR: int = 10):
    """
    Plots the light curve for one object directly in flux space (no magnitude
    conversion), with the pre-peak linear fit for each band overlaid as a dashed
    line. Also prints the fitted slope and number of pre-peak points used, per band,
    and appends the same values to slopes_output_path so they can be inspected after
    a batch run rather than just read off the console.

    Working in flux rather than magnitude avoids fitting a straight line to a
    logarithmic quantity, which was flagged as a potential source of distortion in
    the rate-of-rise filter's output.
    """
    df = df[df["r:psfFlux"] > 0].copy()
    df = df.sort_values("r:midpointMjdTai")
    df["date"] = Time(df["r:midpointMjdTai"].values, format="mjd").to_datetime()

    fig, ax = plt.subplots(figsize=(10, 6))
    fitted_slope_rows = []

    for band, group in df.groupby("r:band"):
        colour = band_colours.get(band, "black")

        ax.errorbar(
            group["date"],
            group["r:psfFlux"],
            yerr=group["r:psfFluxErr"],
            fmt="o",
            color=colour,
            label=f"{band}-band",
            capsize=3,
            markersize=5,
        )



        significant_group_df = group[group["r:psfFlux"] / group["r:psfFluxErr"] > min_peak_SNR]
        if significant_group_df.empty:
            continue
        
        # Fit and overlay the pre-peak slope for this band
        peak_flux = significant_group_df["r:psfFlux"].max()
        peak_mjd = significant_group_df.loc[significant_group_df["r:psfFlux"].idxmax(), "r:midpointMjdTai"]
        pre_peak = significant_group_df[significant_group_df["r:midpointMjdTai"] < peak_mjd]

        if len(pre_peak) < min_pre_peak_points:
            print(f"  [{diaObjectId}] {band}-band: only {len(pre_peak)} pre-peak point(s), skipping fit.")
            continue

        time_span = pre_peak["r:midpointMjdTai"].max() - pre_peak["r:midpointMjdTai"].min()
        if time_span < min_pre_peak_time_span:
            print(f"  [{diaObjectId}] {band}-band: pre-peak points only span {time_span:.2f} days, skipping fit.")
            continue

        normalised_flux = pre_peak["r:psfFlux"] / peak_flux
        normalised_err = (pre_peak["r:psfFluxErr"] / peak_flux).clip(lower=1e-6)
        weights = 1 / normalised_err

        slope, intercept = np.polyfit(pre_peak["r:midpointMjdTai"], normalised_flux, 1, w=weights)

        print(f"  [{diaObjectId}] {band}-band: slope={slope:.5f}, n_pre_peak={len(pre_peak)}")

        fitted_slope_rows.append({
            "diaObjectId": diaObjectId,
            "band": band,
            "slope": slope,
            "n_pre_peak_points": len(pre_peak),
        })

        fit_mjd = pre_peak["r:midpointMjdTai"].sort_values()
        fit_flux = (slope * fit_mjd + intercept) * peak_flux
        fit_dates = Time(fit_mjd.values, format="mjd").to_datetime()

        ax.plot(
            fit_dates,
            fit_flux,
            linestyle="--",
            color=colour,
            alpha=0.7,
            label=f"{band} fit (slope={slope:.4f})",
        )

    ax.set_xlabel("Observation Date")
    ax.set_ylabel("Flux (nanojansky)")
    ax.set_title(f"Light curve - {diaObjectId}\n({source_file})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig(f"{output_dir}.png")
    plt.close(fig)

    append_fitted_slopes(fitted_slope_rows, slopes_output_path)

In [5]:
import pandas as pd
from pathlib import Path

def batch_plotter(object_ids, chunk_size=500, output_path=Path.home() / "Documents/University/Honours/Honours_Project/images/SN_Ia_lightcurves/passed_rate_of_rise_cuts_results"):
    for diaObjectId in range(0, len(object_ids), chunk_size):
        chunk = ",".join(object_ids[diaObjectId:diaObjectId + chunk_size])
        light_curve_info = collect_light_curve_data(chunk)
        if light_curve_info is None or light_curve_info.empty:
            print(f"No light curve data found for chunk starting at index {diaObjectId}.")
            continue
        for object_id, group_df in light_curve_info.groupby("r:diaObjectId"):
            object_id = str(object_id)
            # lightcurve_plotting takes a df, diaObjectId and a source file string as arguments
            lightcurve_plotting(group_df, object_id, object_id, output_path / f"{object_id}")
batch_plotter(FAILED_OBJECTS_DF, 500, Path.home() / "Documents"/"University"/"Honours"/"Honours_Project"/"images"/"test_data_images"/"extragalactic_lt20mag_test" / "fail")

  [170019696450994581] i-band: slope=-0.01210, n_pre_peak=20
  [170019696450994581] r-band: pre-peak points only span 3.05 days, skipping fit.
  [170019717065998487] g-band: only 2 pre-peak point(s), skipping fit.
  [170019717081202796] g-band: only 0 pre-peak point(s), skipping fit.
  [170019717113184373] z-band: only 0 pre-peak point(s), skipping fit.
  [170019717277810735] g-band: pre-peak points only span 0.04 days, skipping fit.
  [170019717277810735] i-band: only 2 pre-peak point(s), skipping fit.
  [170019717277810735] r-band: pre-peak points only span 0.04 days, skipping fit.
  [170019717277810735] y-band: only 1 pre-peak point(s), skipping fit.
  [170019717277810735] z-band: pre-peak points only span 3.00 days, skipping fit.
  [170028485647007777] g-band: only 0 pre-peak point(s), skipping fit.
  [170028485647007777] i-band: only 0 pre-peak point(s), skipping fit.
  [170028485654347961] g-band: only 0 pre-peak point(s), skipping fit.
  [170028485665882165] g-band: only 0 pre-p

In [ ]:
"""
Computes pre-peak slopes (across all available bands) for a known set of SN-like
ground truth objects, using the same compute_all_band_slopes logic as the production
rate-of-rise filter, then plots histograms of the resulting slopes to visually check
whether the distribution looks roughly Gaussian - before deciding whether a
sigma-based significance cut is statistically justified.

Extreme outliers (e.g. slopes in the tens or hundreds, likely from clustered-noise
fits that slipped through) are excluded via a configurable cutoff so they don't
distort the histogram's shape or axis scale.
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# CSV containing the known SN-like ground truth objects. Assumed to have a
# "r:diaObjectId" column, matching the convention used elsewhere in the project -
# update this path/column name if your ground truth CSV differs.
GROUND_TRUTH_PATH = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data" / "sn_like_test_objects.csv"

OUTPUT_SLOPES_PATH = Path.home() / "Pictures" / "sn_like_test_object_slopes.csv"
OUTPUT_HISTOGRAM_PATH = OUTPUT_SLOPES_PATH.parent / "ground_truth_slope_distribution_histograms.png"
OUTPUT_LOG_HISTOGRAM_PATH = OUTPUT_SLOPES_PATH.parent / "ground_truth_log_slope_distribution_histograms.png"

# Slopes with an absolute value beyond this are excluded from the plot as outliers.
OUTLIER_CUTOFF = 1.0

def compute_ground_truth_slopes(object_ids: list[str], chunk_size: int = 500) -> pd.DataFrame:
    """
    Fetches light curve data for the given object IDs and computes the per-band
    pre-peak slope for each object using compute_all_band_slopes, so the slopes
    used for this Gaussianity check are calculated identically to how the
    production filter calculates them.

    Returns a DataFrame with columns: diaObjectId, band, slope, n_pre_peak_points.
    """
    light_curve_info = chunked_light_curve_data(object_ids, chunk_size)

    rows = []
    for object_id, group_df in light_curve_info.groupby("r:diaObjectId"):
        band_slopes = compute_all_band_slopes(group_df)
        for band, (slope, n_points) in band_slopes.items():
            rows.append({
                "diaObjectId": object_id,
                "band": band,
                "slope": slope,
                "n_pre_peak_points": n_points,
            })

    return pd.DataFrame(rows)


def filter_outliers(df: pd.DataFrame, outlier_cutoff: float) -> pd.DataFrame:
    """
    Drops rows beyond the outlier cutoff, printing how many rows were excluded so
    the cutoff's effect is visible rather than silent.
    """
    n_total = len(df)
    df_filtered = df[df["slope"].abs() <= outlier_cutoff]
    n_excluded = n_total - len(df_filtered)

    print(f"Computed {n_total} slope entries, excluded {n_excluded} beyond |slope| > {outlier_cutoff}")

    return df_filtered


def plot_slope_histograms(df: pd.DataFrame, output_path: Path) -> None:
    """
    Plots two views of the slope distribution: one histogram of all slopes combined
    across bands, and one set of per-band histograms, since different bands are
    expected to have different characteristic rise rates.
    """
    bands = sorted(df["band"].unique())

    fig, axes = plt.subplots(1, len(bands) + 1, figsize=(5 * (len(bands) + 1), 5), sharey=False)

    # Combined histogram, all bands together
    axes[0].hist(df["slope"], bins=10, color="steelblue", edgecolor="black")
    axes[0].set_title(f"All bands (n={len(df)})")
    axes[0].set_xlabel("Fitted pre-peak slope")
    axes[0].set_ylabel("Count")
    axes[0].grid(alpha=0.3)

    # Per-band histograms
    for ax, band in zip(axes[1:], bands):
        band_slopes = df[df["band"] == band]["slope"]
        ax.hist(band_slopes, bins=10, color="darkorange", edgecolor="black")
        ax.set_title(f"{band}-band (n={len(band_slopes)})")
        ax.set_xlabel("Fitted pre-peak slope")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Saved histogram figure to {output_path}")

def plot_log_slope_histograms(df: pd.DataFrame, output_path: Path) -> None:
    """
    Plots two views of the slope distribution: one histogram of all slopes, 
    undergoing a log transform combined across bands, and one set of per-band 
    histograms also undergoing a log transform, since different bands are 
    expected to have different characteristic rise rates.
    """
    bands = sorted(df["band"].unique())

    fig, axes = plt.subplots(1, len(bands) + 1, figsize=(5 * (len(bands) + 1), 5), sharey=False)

    # Combined histogram, all bands together
    axes[0].hist(np.log(df["slope"]), bins=10, color="steelblue", edgecolor="black")
    axes[0].set_title(f"All bands (n={len(df)})")
    axes[0].set_xlabel("Fitted pre-peak slope")
    axes[0].set_ylabel("Count")
    axes[0].grid(alpha=0.3)

    # Per-band histograms
    for ax, band in zip(axes[1:], bands):
        band_slopes = df[df["band"] == band]["slope"]
        positive_only = band_slopes[band_slopes > 0]
        log_slopes = np.log(positive_only)
        ax.hist(log_slopes, bins=10, color="darkorange", edgecolor="black")
        ax.set_title(f"{band}-band (n={len(log_slopes)})")
        ax.set_xlabel("Fitted pre-peak slope")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Saved histogram figure to {output_path}")


if __name__ == "__main__":
    ground_truth_ids = list(pd.read_csv(GROUND_TRUTH_PATH, dtype={"diaObjectId": str})["diaObjectId"])

    slopes_df = compute_ground_truth_slopes(ground_truth_ids)
    slopes_df.to_csv(OUTPUT_SLOPES_PATH, index=False)
    print(f"Saved raw ground truth slopes to {OUTPUT_SLOPES_PATH}")

    slopes_df_filtered = filter_outliers(slopes_df, OUTLIER_CUTOFF)
    plot_slope_histograms(slopes_df_filtered, OUTPUT_HISTOGRAM_PATH)
    plot_log_slope_histograms(slopes_df_filtered, OUTPUT_LOG_HISTOGRAM_PATH)

Saved raw ground truth slopes to /home/riley/Pictures/sn_like_test_object_slopes.csv
Computed 180 slope entries, excluded 0 beyond |slope| > 1.0
Saved histogram figure to /home/riley/Pictures/ground_truth_slope_distribution_histograms.png


/home/riley/Documents/University/Honours/Honours_Project/venv/lib64/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Saved histogram figure to /home/riley/Pictures/ground_truth_log_slope_distribution_histograms.png


In [74]:
"""
Computes a 10th-percentile-based pre-peak slope threshold per band from the ground
truth confirmed SNe Ia, then tests that threshold against a small hand-picked batch
of 10 known SN-like and 10 known non-SN-like objects, to see what it excludes and
includes before trusting it on the full candidate population.

This replaces the earlier mean +/- sigma approach, since the ground truth slope
distributions were found to be right-skewed (not Gaussian) and log-transforming
overcorrected into a left skew, so a percentile-based cut was chosen instead as it
makes no assumption about the shape of the distribution.
"""

import pandas as pd
from pathlib import Path

# Slopes computed earlier from the 58 confirmed ground truth Ias (see plot_slope_distributions.py)
GROUND_TRUTH_SLOPES_PATH = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data" / "sn_like_test_object_slopes.csv"

# TODO: confirm these are the correct paths/column names for your 10 SN-like and
# 10 non-SN-like test CSVs.
MIXED_TEST_OBJECTS_PATH = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data" / "test_objects.csv"
NON_SN_LIKE_OBJECTS_PATH = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data" / "non_sn_like_test_objects.csv"
SN_LIKE_OBJECTS_PATH = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data" / "sn_like_test_objects.csv"
PERCENTILE = 35

# Bands with fewer than this many ground truth slope measurements are excluded from
# the percentile-based threshold, since a percentile computed from very few points
# (like u and y, which only had 2-6) isn't statistically meaningful.
MIN_GROUND_TRUTH_SAMPLES = 10


def compute_percentile_thresholds(ground_truth_slopes: pd.DataFrame, percentile: float, min_samples: int) -> dict[str, float]:
    """
    Computes the given percentile of the pre-peak slope distribution for each band,
    using only bands with at least min_samples ground truth measurements.

    Returns a dict of band -> threshold slope. Bands without enough ground truth
    data are omitted, so the caller needs to decide how to handle those bands
    separately (e.g. falling back to a simple slope > 0 check).
    """
    thresholds = {}

    for band, group in ground_truth_slopes.groupby("band"):
        if len(group) < min_samples:
            print(f"Skipping {band}-band threshold: only {len(group)} ground truth samples (need >= {min_samples})")
            continue

        threshold = group["slope"].quantile(percentile / 100)
        thresholds[band] = threshold
        print(f"{band}-band: {percentile}th percentile threshold = {threshold:.5f} (n={len(group)})")

    return thresholds


def object_passes_percentile_cut(group_df: pd.DataFrame, thresholds: dict[str, float]) -> tuple[bool, dict[str, float]]:
    """
    Computes this object's per-band slopes and checks whether at least one band
    exceeds that band's percentile threshold. Bands without a calibrated threshold
    (see MIN_GROUND_TRUTH_SAMPLES) fall back to a simple slope > 0 check.

    Returns (passed, band_slopes) so the caller can inspect the actual slope values
    alongside the pass/fail outcome.
    """
    band_slopes = compute_all_band_slopes(group_df)

    if not band_slopes:
        return False, {}

    for band, (slope, n_points) in band_slopes.items():
        threshold = thresholds.get(band)
        if threshold is not None:
            if slope >= threshold:
                # if len(band_slopes) >= 3:
                    return True, {b: s for b, (s, n) in band_slopes.items()}
        else:
            if slope > 0:
                return True, {b: s for b, (s, n) in band_slopes.items()}

    return False, {b: s for b, (s, n) in band_slopes.items()}


def test_percentile_cut(object_ids: list[str], thresholds: dict[str, float], label: str) -> None:
    """
    Fetches light curve data for the given test objects and prints whether each
    passes the percentile-based cut, along with its per-band slopes, so the results
    can be compared against the known label (SN-like or non-SN-like).
    """
    light_curve_info = chunked_light_curve_data(object_ids)

    print(f"\n--- Testing {label} objects ---")
    for object_id, group_df in light_curve_info.groupby("r:diaObjectId"):
        passed, band_slopes = object_passes_percentile_cut(group_df, thresholds)
        slopes_str = ", ".join(f"{band}={slope:.5f}" for band, slope in band_slopes.items())
        print(f"{object_id}: {'PASSED' if passed else 'FAILED'}  [{slopes_str}]")


if __name__ == "__main__":
    ground_truth_slopes = pd.read_csv(GROUND_TRUTH_SLOPES_PATH, dtype={"diaObjectId": str})
    thresholds = compute_percentile_thresholds(ground_truth_slopes, PERCENTILE, MIN_GROUND_TRUTH_SAMPLES)

    # mixed_test_objects = list(pd.read_csv(MIXED_TEST_OBJECTS_PATH, dtype={"r:diaObjectId": str})["r:diaObjectId"])
    sn_like_objects = list(pd.read_csv(SN_LIKE_OBJECTS_PATH, dtype={"diaObjectId": str})["diaObjectId"])
    # non_sn_like_objects = list(pd.read_csv(NON_SN_LIKE_OBJECTS_PATH, dtype={"diaObjectId": str})["diaObjectId"])

    # test_percentile_cut(mixed_test_objects, thresholds, "Mixed SN and non SN like")
    test_percentile_cut(sn_like_objects, thresholds, "SN Like")
    # test_percentile_cut(non_sn_like_objects, thresholds, "Non SN Like")

g-band: 35th percentile threshold = 0.03489 (n=39)
i-band: 35th percentile threshold = 0.01939 (n=47)
r-band: 35th percentile threshold = 0.02847 (n=46)
Skipping u-band threshold: only 6 ground truth samples (need >= 10)
Skipping y-band threshold: only 2 ground truth samples (need >= 10)
z-band: 35th percentile threshold = 0.01040 (n=40)

--- Testing SN Like objects ---
170019716323082348: PASSED  [r=0.05631, i=0.05941, g=0.06553, z=0.04646, y=0.03242]
170028511473959008: PASSED  [r=0.02237, g=0.02944, i=0.01307, z=0.01161]
170032882395906068: PASSED  [i=0.00340, g=0.05082]
170032900274651160: PASSED  [z=0.04966, r=0.07963, i=0.07043, g=0.09294]
170032917163016282: PASSED  [r=0.18144, i=0.13553, g=0.15364]
170046089928376424: PASSED  [i=0.11396, r=0.08794, z=0.11279, g=0.16494, y=0.04183]
170094456357257301: FAILED  [i=0.00921, r=0.00356]
170094456400773135: FAILED  [i=0.01617, z=0.00694]
170226393328648251: PASSED  [r=0.03748, i=0.04840, z=0.01771, g=0.03847]
170230797176406143: PASSE